# Demo 9 — RAG on your laptop

Today you built the R in RAG: a retriever that counts shared words. This demo
adds the G, a model that writes the answer from what retrieval found.
Everything runs on your machine. No key, no bill, and still no embeddings.

It uses two things you already have:

- `retrieve()`, the baseline from your session 6 notebook, over the course pages
- `answer_question()` from `agent.py`, the pipeline in section 10 of your
  notebook, with a real model where section 10 used a fake one

**You need** Ollama running with `qwen2.5:7b-instruct`
([a local model](https://gecko-academy.github.io/dev3pack-cohort-2026-09/unit0/local-model)).
Without it, this notebook plays one recorded run and says so. Nothing fails.

In [ ]:
# Setup. Works from anywhere inside the course checkout.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise SystemExit(f"No course found above {Path.cwd()}. Open this inside your checkout.")
sys.path.insert(0, str(REPO_ROOT / "src"))
print("ready")

## 1. Which model is answering?

The first live answer takes about 20 seconds while the model loads. After that,
a few seconds each.

In [ ]:
import json

from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.ollama import DEFAULT_MODEL, OllamaClient, probe

RECORDING = json.loads(
    (REPO_ROOT / "demos" / "fixtures" / "rag-recorded.json").read_text(encoding="utf-8")
)

check = probe()
if check.ok:
    model = OllamaClient()
    print(f"[live] {DEFAULT_MODEL}, on this machine")
else:
    model = FakeLLM(responses=RECORDING["responses"])
    made = RECORDING["_provenance"]
    print(f"[recorded] {made['model']}, one run on {made['recorded']}")
    print(f"           to go live: {check.fix}")

## 2. The model alone

Ask about something only this course knows: the ways session 5's mini-agent
can stop. No retrieval, no pages. Just the question.

In [ ]:
QUESTION = "what are the four stop reasons of the mini agent"

print(model.complete(system="Answer in two sentences.", user=QUESTION))

It answers, and it sounds sure. On our recorded run it said *resource
constraints, policy violations, task completion, and safety concerns*. None of
those is right. The model never read this course, so it wrote what a stop
reason usually looks like.

**A fluent answer is not evidence.**

## 3. Retrieval first, then the model

The same question through the pipeline. Retrieval finds three passages. The
model sees only those, and it has to say which page it used.

In [ ]:
from bootcamp_agent.agent import answer_question
from bootcamp_agent.coach import course_documents

PAGES = course_documents()
print(f"{len(PAGES)} course pages")


def run(question, top_k=3):
    result = answer_question(question, PAGES, model, top_k=top_k)
    print()
    for event in result.trace:
        print(f"  trace[{event.kind}] {event.detail}")
    print(f"\n  answer:     {result.answer.answer}")
    print(f"  citations:  {list(result.answer.citations)}")
    print(f"  confidence: {result.answer.confidence}   needs review: {result.answer.needs_human_review}")


run(QUESTION)

Now it is right: `answered`, `budget`, `repeated_call` and `tool_error`, cited to
session 5's introduction. The model did not get smarter. It got the right page.

Count the `llm_call` lines: one.

## 4. Nothing retrieved, so no model call

In [ ]:
run("xylophone quarterly dividend")

No passage shares a word with the question, so the pipeline refuses **before**
it calls the model. There is no `llm_call` line. The part that could invent an
answer is never asked. It is the cheapest refusal there is.

## 5. The wrong pages, and a confident answer

The course has no refund policy. Ask anyway.

In [ ]:
run("what is the refund policy for the bootcamp")

Read the trace first. Retrieval returned three passages, all tied on the same
score, and none of them is about refunds. They share two words with the
question, *bootcamp* and *policy*. The word *refund* appears nowhere in the
course.

On our recorded run the model wrote three sentences about a "policy file",
with **confidence 1.0**. And the citation check passed, because the page it
cited really was retrieved. That check proves the page came back. It cannot
prove the page answers the question.

Run the cell again. A live model sometimes refuses here and sometimes invents.
When retrieval hands it the wrong pages, you are rolling dice.

## What to take away

- **The model is only as right as the pages it gets.** Retrieval decides first.
- **An empty retrieval refuses before the model runs.** Nothing to invent from.
- **A score, a citation and a confidence of 1.0 are three different things.**
  None of them means *correct*. Read the passage.

Tomorrow, session 7 measures the retriever, so you know when to trust it.

## Your turn

Nothing here is marked.

1. **Ask about your own week.** Pick something from a page you read. Does
   `run(...)` find that page?
2. **Ask the model alone the same thing** with `model.complete(...)`. Compare
   the two answers.
3. **Give it less.** `run(QUESTION, top_k=1)`. Does one passage still carry the
   answer?